In [3]:
import sys
from pathlib import Path

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


In [14]:
import pandas as pd

from src.preprocessing.preprocessing import (
    convert_to_numeric,
    remove_sparse_features,
    remove_constant_features,
)

In [15]:
import pandas as pd

file_path = project_root / "data" / "raw" / "HCC_proteomics_Normal.txt"

df = pd.read_csv(file_path, sep="\t")
df.head()

,idx,111,114,124,126,128,132,136,138,142,...,1022,1026,1028,1032,1042,1044,1046,448,536,698
0,ENSG00000000003.15,25.123487,24.792504,24.813667,24.556260,24.454895,24.808779,25.026977,24.389151,24.473884,...,24.755773,24.407858,24.315442,24.453317,24.396582,24.540703,24.918928,24.876518,24.128921,24.552410
1,ENSG00000000419.12,26.441628,26.253745,26.372726,26.092552,26.482692,26.311869,26.234817,26.126119,26.158555,...,26.125231,26.021210,25.968990,26.313476,25.965280,26.123274,26.109980,26.063115,25.605649,26.076056
2,ENSG00000000457.14,23.319190,23.337303,23.175404,23.405809,23.226412,23.402456,23.529123,23.450332,23.286908,...,23.176737,23.071129,23.280037,23.063194,22.999710,23.304137,23.404143,22.993107,22.830460,22.914256
3,ENSG00000000460.17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ENSG00000000938.13,21.392391,21.425340,21.522826,21.598335,21.432756,21.201646,21.367521,21.457526,21.352120,...,20.830519,21.417127,21.410856,20.710182,20.860214,21.412469,21.509806,21.002283,21.438820,21.669404


In [16]:
original_shape = df.shape

df_numeric = convert_to_numeric(df)
df_no_empty = remove_sparse_features(df_numeric)
df_preprocessed = remove_constant_features(df_no_empty)

print("Original shape:", original_shape)
print("After numeric conversion:", df_numeric.shape)
print("After removing sparse features:", df_no_empty.shape)
print("After removing constant features:", df_preprocessed.shape)

Original shape: (11175, 166)
After numeric conversion: (11175, 166)
After removing sparse features: (8368, 166)
After removing constant features: (8368, 166)


In [17]:
import importlib
import src.preprocessing.preprocessing as prep

importlib.reload(prep)

print(dir(prep))

['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'convert_to_numeric', 'missingness_abundance_correlation', 'missingness_report', 'pd', 'remove_constant_features', 'remove_sparse_features']


In [18]:
import pandas as pd

from src.preprocessing.preprocessing import missingness_report

feature_missingness = missingness_report(df_numeric)

feature_missingness.head(20)

,idx,missing_count,missing_percentage
0,ENSG00000116132.12,161,97.575758
1,ENSG00000116039.13,161,97.575758
2,ENSG00000131914.11,161,97.575758
3,ENSG00000164816.8,161,97.575758
4,ENSG00000120057.5,161,97.575758
5,ENSG00000238074.5,161,97.575758
6,ENSG00000118849.10,161,97.575758
7,ENSG00000073067.14,161,97.575758
8,ENSG00000196368.5,161,97.575758
9,ENSG00000282815.1,161,97.575758


In [7]:
import importlib
import src.preprocessing.preprocessing as prep

importlib.reload(prep)

<module 'src.preprocessing.preprocessing' from 'c:\\Users\\lebar0040\\OneDrive - University of Bergen\\ai-analytics-agent\\src\\preprocessing\\preprocessing.py'>

In [20]:
df_filtered = prep.remove_sparse_features(
    df_numeric,
    max_missing_percentage=30.0,
)

print("Original number of features:", df_numeric.shape[0])
print("Features after 30% missingness filter:", df_filtered.shape[0])
print("Features removed:", df_numeric.shape[0] - df_filtered.shape[0])

Original number of features: 11175
Features after 30% missingness filter: 8368
Features removed: 2807


In [21]:
filtered_missingness = prep.missingness_report(df_filtered)

print(
    "Maximum missingness after filtering:",
    filtered_missingness["missing_percentage"].max()
)

Maximum missingness after filtering: 28.484848484848484


In [22]:

## calculate 
# how much data remains missing before imputation


sample_columns = [
    column for column in df_filtered.columns
    if column != "idx"
]

remaining_missing_count = (
    df_filtered[sample_columns]
    .isna()
    .sum()
    .sum()
)

remaining_missing_percentage = (
    remaining_missing_count
    / df_filtered[sample_columns].size
    * 100
)

print("Remaining missing values:", remaining_missing_count)
print(
    f"Remaining missing percentage: "
    f"{remaining_missing_percentage:.2f}%"
)

Remaining missing values: 32725
Remaining missing percentage: 2.37%


In [23]:
filter_summary = {
    "original_features": df_numeric.shape[0],
    "retained_features": df_filtered.shape[0],
    "removed_features": df_numeric.shape[0] - df_filtered.shape[0],
    "retained_percentage": round(
        100 * df_filtered.shape[0] / df_numeric.shape[0], 2
    ),
    "missingness_threshold": 30.0,
    "remaining_missing_values": remaining_missing_count,
    "remaining_missing_percentage": round(
        remaining_missing_percentage, 2
    ),
}

filter_summary

{'original_features': 11175,
 'retained_features': 8368,
 'removed_features': 2807,
 'retained_percentage': 74.88,
 'missingness_threshold': 30.0,
 'remaining_missing_values': np.int64(32725),
 'remaining_missing_percentage': np.float64(2.37)}

In [24]:
feature_missingness = (
    df_filtered
    .drop(columns=["idx"])
    .isna()
    .mean(axis=1)
    * 100
)

feature_missingness.describe()

count    8368.000000
mean        2.370140
std         5.906298
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max        28.484848
dtype: float64

In [25]:
bins = [-0.01, 0, 5, 10, 20, 30]

pd.cut(
    feature_missingness,
    bins=bins,
    labels=[
        "0%",
        "0–5%",
        "5–10%",
        "10–20%",
        "20–30%"
    ]
).value_counts().sort_index()

0%        6660
0–5%       449
5–10%      478
10–20%     427
20–30%     354
Name: count, dtype: int64

In [26]:
#Missingness across samples
sample_data = df_filtered.drop(columns=["idx"])

sample_missingness = (
    sample_data
    .isna()
    .mean(axis=0)
    * 100
)

sample_missingness.describe()

count    165.000000
mean       2.370140
std        0.888629
min        0.944073
25%        1.732792
50%        2.210803
75%        3.142925
max        4.027247
dtype: float64

In [27]:
sample_missingness.sort_values(
    ascending=False
).head(10)

642    4.027247
618    4.015296
636    4.015296
616    4.003346
628    4.003346
222    3.967495
214    3.955545
192    3.955545
196    3.955545
218    3.955545
dtype: float64

In [28]:
#Check missingness versus protein abundance
protein_mean = sample_data.mean(axis=1)

missingness_vs_abundance = pd.DataFrame({
    "mean_abundance": protein_mean,
    "missing_percentage": feature_missingness
})

missingness_vs_abundance.corr(method="spearman")

,mean_abundance,missing_percentage
mean_abundance,1.000000,-0.605323
missing_percentage,-0.605323,1.000000


In [29]:
print("Total dataframe columns:", df_filtered.shape[1])
print(
    "Sample columns:",
    len([c for c in df_filtered.columns if c != "idx"])
)

print(df_filtered.columns[:10].tolist())
print(df_filtered.columns[-10:].tolist())

Total dataframe columns: 166
Sample columns: 165
['idx', '111', '114', '124', '126', '128', '132', '136', '138', '142']
['1022', '1026', '1028', '1032', '1042', '1044', '1046', '448', '536', '698']


In [31]:
#import importlib
#import src.preprocessing.preprocessing as prep

#importlib.reload(prep)

In [30]:
spearman_corr = prep.missingness_abundance_correlation(
    df_filtered,
    id_column="idx",
)

print(
    "Spearman correlation between abundance and missingness:",
    round(spearman_corr, 3),
)

Spearman correlation between abundance and missingness: -0.605


In [33]:
output_path = (
    project_root
    / "data"
    / "interim"
    / "HCC_proteomics_filtered.csv"
)

df_filtered.to_csv(
    output_path,
    index=False
)

print(f"Saved filtered data to: {output_path}")

Saved filtered data to: c:\Users\lebar0040\OneDrive - University of Bergen\ai-analytics-agent\data\interim\HCC_proteomics_filtered.csv
